In [ ]:
# Ячейка 1: Постановка
"""
## Модернизация ремонтной базы

Primal: распределение средств на ремонт техники X и оборудования Y.
Боеготовность: 100 ед за ремонт техники, 150 ед за оборудование.

Ограничения:
- Механики: 3X + 5Y <= 150
- Запчасти: 4X + 2Y <= 120
- Время: X + 3Y <= 90
X, Y >= 0

Задание:
1. Решить primal
2. Какие ресурсы дефицитны
3. Теневые цены ресурсов
4. Что если добавить 5 механиков
5. Что если убрать 10 запчастей
"""

In [ ]:
# Ячейка 2: Решение и анализ
import numpy as np
from scipy.optimize import linprog

c = [-100, -150]
A = [[3, 5], [4, 2], [1, 3]]
b = [150, 120, 90]

result = linprog(c, A_ub=A, b_ub=b, bounds=(0, None), method='highs')
print(f"Техника X={result.x[0]:.2f}, Оборудование Y={result.x[1]:.2f}")
print(f"Боеготовность={-result.fun:.2f}")

# Запасы
slack = b - A @ result.x
for i, (name, s) in enumerate(zip(['Механики','Запчасти','Время'], slack)):
    print(f"{name}: запас={s:.2f}, {'ДЕФИЦИТ' if s < 0.001 else 'избыток'}")

# Dual
c_dual = [150, 120, 90]
A_dual = [[-3, -4, -1], [-5, -2, -3]]
b_dual = [-100, -150]
dual = linprog(c_dual, A_ub=A_dual, b_ub=b_dual, bounds=(0, None), method='highs')

print(f"\nТеневые цены:")
print(f"Механики: {dual.x[0]:.1f} ед/чел")
print(f"Запчасти: {dual.x[1]:.1f} ед/ед")
print(f"Время: {dual.x[2]:.1f} ед/час")

# Прогноз: +5 механиков
b_new = [155, 120, 90]
result_new = linprog(c, A_ub=A, b_ub=b_new, bounds=(0, None), method='highs')
delta = -result_new.fun + result.fun
print(f"\nПрогноз +5 механиков: +{5*dual.x[0]:.1f}")
print(f"Реальность: +{delta:.1f}")

# Прогноз: -10 запчастей
b_new2 = [150, 110, 90]
result_new2 = linprog(c, A_ub=A, b_ub=b_new2, bounds=(0, None), method='highs')
delta2 = -result_new2.fun + result.fun
print(f"\nПрогноз -10 запчастей: {10*dual.x[1]:.1f} (потеря)")
print(f"Реальность: {delta2:.1f}")